# Extract Gene Embeddings with Geneformer

Extract gene embeddings from `Immune_ALL_human.h5ad` using the pretrained
Geneformer model and visualize with UMAP.

In [1]:
import pickle
from pathlib import Path

import numpy as np
import scanpy as sc
import torch
from geneformer import ENSEMBL_DICTIONARY_FILE, EmbExtractor, TranscriptomeTokenizer

DATA_DIR = Path("../data")
TOKENIZED_DIR = DATA_DIR / "geneformer_tokenized"
OUTPUT_DIR = DATA_DIR / "geneformer_output"
MODEL_NAME = "ctheodoris/Geneformer"

TOKENIZED_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"CUDA available: {torch.cuda.is_available()}")

/home/chasty2/Documents/scFM_benchmarking/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available: True


## Load and inspect data

In [2]:
adata = sc.read(DATA_DIR / "Immune_ALL_human.h5ad")
print(f"Dataset: {adata.shape[0]} cells, {adata.shape[1]} genes")
print(f"var columns: {list(adata.var.columns)}")
print(f"obs columns: {list(adata.obs.columns)}")
print(f"First 5 var_names: {list(adata.var_names[:5])}")

Dataset: 33506 cells, 12303 genes
var columns: []
obs columns: ['batch', 'chemistry', 'data_type', 'dpt_pseudotime', 'final_annotation', 'mt_frac', 'n_counts', 'n_genes', 'sample_ID', 'size_factors', 'species', 'study', 'tissue']
First 5 var_names: ['LINC00115', 'FAM41C', 'SAMD11', 'NOC2L', 'KLHL17']


## Map gene names to Ensembl IDs

Geneformer requires `ensembl_id` in `adata.var` and `n_counts` in `adata.obs`.
We use geneformer's bundled gene name → Ensembl ID dictionary for mapping.

In [3]:
# Load geneformer's gene name → Ensembl ID dictionary
with open(ENSEMBL_DICTIONARY_FILE, "rb") as f:
    gene_name_to_ensembl = pickle.load(f)

print(f"Genes in mapping dict: {len(gene_name_to_ensembl)}")

# Map var_names to Ensembl IDs
ensembl_ids = [gene_name_to_ensembl.get(gene, "unknown") for gene in adata.var_names]
adata.var["ensembl_id"] = ensembl_ids

mapped = sum(1 for eid in ensembl_ids if eid != "unknown")
print(f"Mapped {mapped}/{len(ensembl_ids)} genes to Ensembl IDs")

# Ensure n_counts exists
if "n_counts" not in adata.obs.columns:
    adata.obs["n_counts"] = np.array(adata.X.sum(axis=1)).flatten()
    print("Computed n_counts from raw counts")
else:
    print("n_counts already present")

# Identify highly variable genes (matching scGPT's N_HVG=1200)
adata_hvg = adata.copy()
sc.pp.normalize_total(adata_hvg)
sc.pp.log1p(adata_hvg)
sc.pp.highly_variable_genes(adata_hvg, n_top_genes=1200)
hvg_names = set(adata_hvg.var_names[adata_hvg.var.highly_variable])
hvg_ensembl = {gene_name_to_ensembl[g] for g in hvg_names if g in gene_name_to_ensembl}
print(f"HVGs selected: {len(hvg_names)}, mapped to Ensembl: {len(hvg_ensembl)}")
del adata_hvg

# Save prepared h5ad for tokenization (tokenizer reads from a directory)
PREP_DIR = DATA_DIR / "geneformer_prep"
PREP_DIR.mkdir(exist_ok=True)
prep_path = PREP_DIR / "Immune_ALL_human.h5ad"
adata.write(prep_path)
print(f"Saved prepared h5ad to {prep_path}")

Genes in mapping dict: 63675
Mapped 12014/12303 genes to Ensembl IDs
n_counts already present
HVGs selected: 1200, mapped to Ensembl: 1177
Saved prepared h5ad to ../data/geneformer_prep/Immune_ALL_human.h5ad


## Tokenize with TranscriptomeTokenizer

In [4]:
tk = TranscriptomeTokenizer(
    custom_attr_name_dict={"final_annotation": "cell_type"},
    nproc=4,
)
tk.tokenize_data(
    data_directory=PREP_DIR,
    output_directory=TOKENIZED_DIR,
    output_prefix="immune_human",
    file_format="h5ad",
)
print(f"Tokenized dataset saved to {TOKENIZED_DIR / 'immune_human.dataset'}")

Tokenizing ../data/geneformer_prep/Immune_ALL_human.h5ad


/home/chasty2/Documents/scFM_benchmarking/.venv/lib/python3.12/site-packages/geneformer/tokenizer.py:544: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  for i in adata.var["ensembl_id_collapsed"][coding_miRNA_loc]
/home/chasty2/Documents/scFM_benchmarking/.venv/lib/python3.12/site-packages/geneformer/tokenizer.py:547: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  coding_miRNA_ids = adata.var["ensembl_id_collapsed"][coding_miRNA_loc]


../data/geneformer_prep/Immune_ALL_human.h5ad has no column attribute 'filter_pass'; tokenizing all cells.
Creating dataset.
Tokenized dataset saved to ../data/geneformer_tokenized/immune_human.dataset


## Extract gene embeddings

In [ ]:
embex = EmbExtractor(
    model_type="Pretrained",
    emb_mode="gene",
    max_ncells=1000,
    emb_layer=-1,
    forward_batch_size=10,
    summary_stat="mean",
)

tokenized_dataset_path = TOKENIZED_DIR / "immune_human.dataset"

embs_df = embex.extract_embs(
    model_directory=MODEL_NAME,
    input_data_file=tokenized_dataset_path,
    output_directory=OUTPUT_DIR,
    output_prefix="gene_embs",
)

print(f"All gene embeddings: {embs_df.shape}")

# Filter to highly variable genes
embs_df = embs_df.loc[embs_df.index.isin(hvg_ensembl)]
print(f"After HVG filter: {embs_df.shape}")
embs_df.head()

BertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
/home/chasty2/Documents/scFM_benchmarking/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:777: UserWarning: `return_dict_in_genera

OutOfMemoryError: CUDA out of memory. Tried to allocate 18.00 MiB. GPU 

## Label genes with gene symbols

Map Ensembl IDs back to human-readable gene symbols.

In [ ]:
# Build reverse mapping: Ensembl ID → gene name
ensembl_to_gene_name = {v: k for k, v in gene_name_to_ensembl.items()}

# Label each gene embedding with its symbol
gene_symbols = [ensembl_to_gene_name.get(eid, eid) for eid in embs_df.index]
embs_df.insert(0, "gene_symbol", gene_symbols)

print(f"Labeled {sum(s != e for s, e in zip(gene_symbols, embs_df.index))}/{len(gene_symbols)} genes with symbols")
embs_df[["gene_symbol"]].head(10)

## UMAP of gene embeddings

In [ ]:
# Build AnnData from gene embeddings for scanpy UMAP
emb_cols = [c for c in embs_df.columns if c != "gene_symbol"]
gene_adata = sc.AnnData(
    X=embs_df[emb_cols].values.astype(np.float32),
    obs={"gene_symbol": embs_df["gene_symbol"].values},
)
gene_adata.obs_names = embs_df.index  # Ensembl IDs

sc.pp.neighbors(gene_adata, use_rep="X")
sc.tl.umap(gene_adata)
sc.pl.umap(gene_adata, title="Geneformer Gene Embeddings")